# 05. Neural Network Development (MLP)

## 1. Tổng quan (Overview)
Trong notebook này, chúng ta sẽ triển khai mô hình **Multi-Layer Perceptron (MLP)** - một dạng mạng Nơ-ron nhân tạo sâu (Deep Learning) để giải quyết bài toán dự đoán trầm cảm. 

**Tại sao sử dụng Neural Network cho dữ liệu bảng?**
Mặc dù các mô hình Boosting thường chiếm ưu thế trên dữ liệu bảng, Neural Network với kiến trúc MLP lại có khả năng học được các tương tác phi tuyến tính (non-linear interactions) cực kỳ phức tạp giữa các đặc trưng mà các mô hình truyền thống có thể bỏ sót.

**Chiến lược thực hiện:**
- Sử dụng framework **PyTorch**.
- Tận dụng bộ dữ liệu đã được **Chuẩn hóa (Scaled)** và **Nội suy (Imputed)** từ bước tiền xử lý để đảm bảo tốc độ hội tụ của thuật toán Gradient Descent.
- Thiết lập luồng dữ liệu chuẩn thông qua PyTorch `Dataset` và `DataLoader`.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Cấu hình đường dẫn
INPUT_DIR = '../outputs'
OUTPUT_DIR = '../outputs/artifacts'

os.makedirs(OUTPUT_DIR, exist_ok=True)
# 1. Load Meta Information để đồng bộ hóa tham số
with open(f"{INPUT_DIR}/meta_info.pkl", "rb") as f:
    meta = pickle.load(f)

RANDOM_STATE = meta['random_seed']
N_FOLDS = meta['n_folds']
# Trọng số để xử lý mất cân bằng lớp (Dùng cho Weighted BCE Loss)
POS_WEIGHT = torch.tensor([meta['scale_pos_weight']]) 

print(f"Cấu hình đã nạp: Seed={RANDOM_STATE}, Folds={N_FOLDS}")
print(f"Trọng số lớp dương (Positive Weight): {POS_WEIGHT.item():.4f}")

# Thiết lập thiết bị tính toán (GPU nếu có, không thì dùng CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")

## 2. Chuẩn bị Dữ liệu (Data Preparation)

Chúng ta sẽ sử dụng bộ dữ liệu `train_nn_ready.pkl` - đây là biến thể dữ liệu đã được xử lý bằng **RobustScaler** và **MICE Imputer**, vốn là môi trường lý tưởng cho các hàm kích hoạt (Activation Functions) của mạng Nơ-ron hoạt động hiệu quả.

**Lưu ý quan trọng:** Chúng ta sẽ loại bỏ cột `KFOLD` nếu nó tồn tại để tránh rò rỉ dữ liệu (Data Leakage), vì đây chỉ là biến phục vụ chia nếp gấp, không phải là đặc trưng học máy.

In [ ]:
# 1. Tải dữ liệu đặc trưng và nhãn
X_all = pd.read_pickle(f"{INPUT_DIR}/train_nn_ready.pkl")
y_all = pd.read_pickle(f"{INPUT_DIR}/y_train.pkl")

# 2. Loại bỏ cột metadata KFOLD nếu có
if 'KFOLD' in X_all.columns:
    X_all = X_all.drop(columns=['KFOLD'])
    print("Đã loại bỏ cột KFOLD khỏi dữ liệu huấn luyện.")

# 3. Phân tách tập Train (80%) và Validation (20%)
# Sử dụng stratify=y_all để giữ nguyên tỷ lệ lớp nhãn
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, 
    test_size=0.20, 
    stratify=y_all, 
    random_state=RANDOM_STATE
)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Validation: {X_val.shape}")

## 3. Khởi tạo PyTorch Dataset

Mạng Nơ-ron trong PyTorch không thể đọc trực tiếp định dạng Pandas DataFrame. Chúng ta cần đóng gói dữ liệu vào một lớp `Dataset` tùy chỉnh để chuyển đổi các đặc trưng và nhãn sang dạng **Tensors (Float32)**. Việc này cũng giúp tối ưu hóa bộ nhớ khi huấn luyện theo từng lô (Batch Training).

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        # Chuyển đổi dữ liệu sang Tensor Float32
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Khởi tạo các đối tượng Dataset cho Train và Val
train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)

print("Đã khởi tạo thành công luồng dữ liệu PyTorch Dataset.")

## 4. Xây dựng Kiến trúc Mạng Nơ-ron (MLP Architecture)

Chúng ta sẽ thiết kế một mạng **Multi-Layer Perceptron (MLP)** với 3 lớp ẩn (Hidden Layers). Kiến trúc này được tối ưu hóa để xử lý dữ liệu bảng thông qua các kỹ thuật bổ trợ:

- **Batch Normalization (BN):** Giúp chuẩn hóa đầu ra của các lớp ẩn, tăng tốc độ hội tụ và giảm độ nhạy với việc khởi tạo trọng số.
- **Hàm kích hoạt ReLU:** Giúp mô hình học được các đặc trưng phi tuyến tính phức tạp.
- **Dropout (p=0.3):** Một kỹ thuật Regularization giúp ngăn chặn hiện tượng Overfitting bằng cách ngắt ngẫu nhiên 30% kết nối trong quá trình huấn luyện.
- **Hàm Sigmoid ở lớp cuối:** Chuyển đổi đầu ra thành xác suất (0 đến 1) cho bài toán phân loại nhị phân.

In [ ]:
class DepressionMLP(nn.Module):
    def __init__(self, input_dim):
        super(DepressionMLP, self).__init__()
        
        # 1. Lớp đầu vào và Batch Normalization đầu tiên
        self.bn_input = nn.BatchNorm1d(input_dim)
        
        # 2. Lớp ẩn thứ nhất: input_dim -> 256
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        
        # 3. Lớp ẩn thứ hai: 256 -> 128
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        
        # 4. Lớp ẩn thứ ba: 128 -> 64
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        
        # 5. Lớp đầu ra (Output): 64 -> 1
        self.fc_out = nn.Linear(64, 1)
        
        # Các thành phần bổ trợ
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Input -> BN
        x = self.bn_input(x)
        
        # Lớp 1: Dense(256) -> BN -> ReLU -> Dropout
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        
        # Lớp 2: Dense(128) -> BN -> ReLU -> Dropout
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        
        # Lớp 3: Dense(64) -> BN -> ReLU -> Dropout
        x = self.dropout(self.relu(self.bn3(self.fc3(x))))
        
        # # Output: Dense(1) -> Sigmoid
        # x = self.sigmoid(self.fc_out(x))
        x = self.fc_out(x)
        
        return x

## 5. Khởi tạo và Kiểm tra Kiến trúc (Model Summary)

Chúng ta tiến hành khởi tạo mô hình dựa trên số lượng đặc trưng đầu vào (`input_dim`) đã chuẩn bị ở Phần 1 và chuyển mô hình vào thiết bị tính toán (CPU/GPU). Việc in cấu trúc mô hình giúp xác nhận các lớp đã được thiết lập đúng như thiết kế.

In [ ]:
# Xác định số lượng đặc trưng đầu vào
input_dim = X_train.shape[1]

# Khởi tạo mô hình
model = DepressionMLP(input_dim).to(device)

# Hiển thị cấu trúc mô hình
print("--- KIẾN TRÚC MÔ HÌNH MLP ---")
print(model)

# Kiểm tra thử với một lô dữ liệu nhỏ (Dummy input)
dummy_input = torch.randn(1, input_dim).to(device)
with torch.no_grad():
    output = model(dummy_input)
    print(f"\nDữ liệu đầu ra thử nghiệm (Xác suất): {output.item():.4f}")

## 6. Cấu hình Huấn luyện & Cơ chế Dừng sớm (Early Stopping)

Để đảm bảo mô hình hội tụ tốt nhất mà không bị Overfitting, chúng ta bám sát cấu hình sau:
- **Optimizer:** `AdamW` (Adam với Weight Decay) giúp phạt các trọng số quá lớn (`weight_decay=1e-4`), tỷ lệ học `lr=1e-3`.
- **Scheduler:** `CosineAnnealingLR` giúp giảm dần learning rate theo hình chuông, giúp mô hình "hạ cánh" mượt mà vào điểm cực tiểu của hàm Loss.
- **Loss Function:** `BCEWithLogitsLoss` kết hợp `pos_weight` để phạt nặng hơn nếu mô hình đoán sai lớp thiểu số (Depression).
- **Early Stopping:** Theo dõi Validation Loss. Nếu sau 15 epochs (`patience=15`) mà loss không giảm, tiến trình sẽ tự động dừng và khôi phục lại bộ trọng số (weights) tốt nhất.

In [ ]:
class EarlyStopping:
    """
    Theo dõi Validation Loss, nếu không giảm sau 'patience' epochs thì dừng sớm.
    Đồng thời tự động lưu lại model có Val Loss thấp nhất.
    """
    def __init__(self, patience=15, delta=0.0, path=f'{OUTPUT_DIR}/best_mlp_checkpoint.pt'):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        # Lần lặp đầu tiên
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        # Nếu loss bị tăng lên (tệ đi)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        # Nếu loss tiếp tục giảm (tốt lên)
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Lưu model khi validation loss giảm."""
        # Chỉ in ra khi test 1 fold, trong CV loop ta có thể ẩn đi cho đỡ rối
        # print(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model ...")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

## 7. Khởi tạo Hàm Train & Evaluate cho từng Epoch

Mạng Nơ-ron yêu cầu định nghĩa rõ ràng vòng lặp Forward Pass và Backward Pass. Chúng ta sẽ viết 2 hàm trợ trợ:
- `train_epoch`: Đẩy dữ liệu qua model, tính Loss, và cập nhật trọng số (Backward step).
- `eval_epoch`: Đóng băng trọng số (`torch.no_grad()`), tính toán Validation Loss và Validation AUC-ROC.

In [ ]:
from sklearn.metrics import roc_auc_score

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Bật chế độ train (kích hoạt Dropout & BatchNorm)
    total_loss = 0.0
    
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Xóa gradient cũ
        optimizer.zero_grad()
        
        # Forward pass (Dự đoán)
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass (Tính đạo hàm & cập nhật trọng số)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_X.size(0)
        
    return total_loss / len(dataloader.dataset)

def eval_epoch(model, dataloader, criterion, device):
    model.eval() # Bật chế độ đánh giá (tắt Dropout, đóng băng BatchNorm)
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            total_loss += loss.item() * batch_X.size(0)
            
            # Chuyển logit thành xác suất bằng sigmoid để tính AUC
            probs = torch.sigmoid(predictions).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(batch_y.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader.dataset)
    auc_score = roc_auc_score(all_targets, all_preds)
    
    return avg_loss, auc_score

## 8. Huấn luyện với 5-Fold Stratified Cross-Validation

Để có cái nhìn khách quan về hiệu năng của Mạng Nơ-ron, chúng ta sẽ thực hiện đánh giá chéo 5 nếp gấp (5-Fold CV) trên tập Huấn luyện (80%). 

**Quy trình thực hiện trong mỗi nếp gấp (Fold):**
1. Chia tập Train thành 2 phần: **Sub-train** để học và **Hold-out fold** để kiểm chứng.
2. Khởi tạo mới hoàn toàn trọng số mô hình, bộ tối ưu hóa và bộ lập lịch.
3. Chạy vòng lặp huấn luyện tối đa 150 Epochs, kết hợp với **Early Stopping** để dừng lại ngay khi mô hình có dấu hiệu Overfitting (sau 15 epochs không cải thiện).
4. Lưu lại xác suất dự đoán của từng Fold để tạo mảng **OOF Predictions**.
5. Tính toán **Mean AUC** và **Std AUC** để đo lường độ ổn định của kiến trúc MLP.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# 1. Khởi tạo các biến lưu trữ kết quả CV
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_probs = np.zeros(len(X_train))
cv_auc_scores = []

# Để indexing X_train, y_train dễ dàng hơn bằng iloc
X_tr_reset = X_train.reset_index(drop=True)
y_tr_reset = y_train.reset_index(drop=True)

print(f"Bắt đầu quy trình {N_FOLDS}-Fold CV cho Neural Network...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr_reset, y_tr_reset), 1):
    print(f"\n--- [FOLD {fold}/{N_FOLDS}] ---")
    
    # Chia dữ liệu cho từng Fold
    x_train_fold, x_val_fold = X_tr_reset.iloc[train_idx], X_tr_reset.iloc[val_idx]
    y_train_fold, y_val_fold = y_tr_reset.iloc[train_idx], y_tr_reset.iloc[val_idx]
    
    # Khởi tạo Dataloaders
    train_fold_ds = TabularDataset(x_train_fold, y_train_fold)
    val_fold_ds = TabularDataset(x_val_fold, y_val_fold)
    
    train_loader = DataLoader(train_fold_ds, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_fold_ds, batch_size=256, shuffle=False)
    
    # Khởi tạo Model, Criterion, Optimizer, Scheduler và EarlyStopping mới cho mỗi Fold
    model_fold = DepressionMLP(input_dim).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(device))
    optimizer = optim.AdamW(model_fold.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    
    checkpoint_path = f"{OUTPUT_DIR}/mlp_fold_{fold}.pt"
    early_stopping = EarlyStopping(patience=15, path=checkpoint_path)
    
    # Vòng lặp Epochs
    for epoch in range(1, 151):
        tr_loss = train_epoch(model_fold, train_loader, criterion, optimizer, device)
        va_loss, va_auc = eval_epoch(model_fold, val_loader, criterion, device)
        
        scheduler.step()
        early_stopping(va_loss, model_fold)
        
        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d}: Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | Val AUC: {va_auc:.4f}")
            
        if early_stopping.early_stop:
            print(f"Dừng sớm tại Epoch {epoch}. Khôi phục trọng số tốt nhất.")
            break
            
    # Load lại trọng số tốt nhất của Fold này để dự đoán OOF
    model_fold.load_state_dict(torch.load(checkpoint_path))
    model_fold.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch_X, _ in val_loader:
            batch_X = batch_X.to(device)
            outputs = model_fold(batch_X)
            probs = torch.sigmoid(outputs).cpu().numpy()
            fold_preds.extend(probs.flatten())
            
    oof_probs[val_idx] = fold_preds
    fold_final_auc = roc_auc_score(y_val_fold, fold_preds)
    cv_auc_scores.append(fold_final_auc)
    print(f"KẾT THÚC FOLD {fold} - AUC Đạt được: {fold_final_auc:.4f}")

# Tính toán kết quả CV tổng thể
print("\n" + "="*40)
print(f"KẾT QUẢ CV {N_FOLDS}-FOLD (TRÊN 80% TRAIN SET):")
print(f"Mean AUC: {np.mean(cv_auc_scores):.4f} (+/- {np.std(cv_auc_scores):.4f})")
print("="*40)

# Lưu OOF Probs để dùng cho Ensemble sau này
np.save(f"{OUTPUT_DIR}/nn_oof_probs.npy", oof_probs)

## 9. Đánh giá trên tập Validation (Hold-out 20%)

Sau khi xác nhận độ ổn định của cấu trúc mạng qua 5-Fold CV, chúng ta sẽ tiến hành **Refit (Huấn luyện lại)** mô hình trên toàn bộ tập Huấn luyện (80%). Việc này giúp mô hình học được nhiều pattern nhất có thể trước khi đối mặt với tập Kiểm chứng (20%).

Các tham số và cơ chế Early Stopping vẫn được giữ nguyên như quá trình Cross-Validation.

In [ ]:
print("Bắt đầu Huấn luyện lại (Refit) trên toàn bộ 80% tập Train...")

# 1. Khởi tạo DataLoaders cho toàn bộ Train và Val
train_ds_full = TabularDataset(X_train, y_train)
val_ds_full = TabularDataset(X_val, y_val)

train_loader_full = DataLoader(train_ds_full, batch_size=256, shuffle=True)
val_loader_full = DataLoader(val_ds_full, batch_size=256, shuffle=False)

# 2. Khởi tạo lại Model và các thành phần tối ưu
final_model = DepressionMLP(input_dim).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(device))
optimizer = optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)

checkpoint_final = f"{OUTPUT_DIR}/best_mlp_final.pt"
early_stopping_final = EarlyStopping(patience=15, path=checkpoint_final)

# 3. Vòng lặp Huấn luyện
for epoch in range(1, 151):
    tr_loss = train_epoch(final_model, train_loader_full, criterion, optimizer, device)
    va_loss, va_auc = eval_epoch(final_model, val_loader_full, criterion, device)
    
    scheduler.step()
    early_stopping_final(va_loss, final_model)
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | Val AUC: {va_auc:.4f}")
        
    if early_stopping_final.early_stop:
        print(f"Dừng sớm tại Epoch {epoch}. Đã lưu trọng số tốt nhất.")
        break

# 4. Tải trọng số tốt nhất và Dự đoán trên tập Validation
final_model.load_state_dict(torch.load(checkpoint_final))
final_model.eval()

val_probs_nn = []
with torch.no_grad():
    for batch_X, _ in val_loader_full:
        batch_X = batch_X.to(device)
        outputs = final_model(batch_X)
        probs = torch.sigmoid(outputs).cpu().numpy()
        val_probs_nn.extend(probs.flatten())

val_probs_nn = np.array(val_probs_nn)
print(f"Hoàn tất dự đoán trên tập Validation. Kích thước mảng dự đoán: {val_probs_nn.shape}")

## 10. Phân tích Chẩn đoán (Diagnostic Analysis) & Trực quan hóa

Để so sánh công bằng với các mô hình Baseline, chúng ta sẽ:
1. Vẽ **Đường cong ROC** và tính điểm **Validation AUC**.
2. Hiển thị **Classification Report** và vẽ **Confusion Matrix** tại ngưỡng mặc định 0.5 để kiểm tra khả năng bắt lỗi (Recall) và báo động giả (Precision).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_curve, auc, f1_score, accuracy_score, 
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# 1. Tính toán các Metrics cơ bản tại ngưỡng 0.5
val_preds_nn = (val_probs_nn >= 0.5).astype(int)
val_auc_nn = roc_auc_score(y_val, val_probs_nn)
val_f1_nn = f1_score(y_val, val_preds_nn)
val_acc_nn = accuracy_score(y_val, val_preds_nn)

print(f"--- KẾT QUẢ TRÊN TẬP VALIDATION (20%) ---")
print(f"Validation AUC : {val_auc_nn:.4f}")
print(f"Validation F1  : {val_f1_nn:.4f}")
print(f"Validation Acc : {val_acc_nn:.4f}")
print("\n" + "="*50)
print("[Classification Report - Ngưỡng 0.5]")
print(classification_report(y_val, val_preds_nn, target_names=['No Depression', 'Depression']))

# Khởi tạo khung hình
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2. Vẽ Confusion Matrix
cm = confusion_matrix(y_val, val_preds_nn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Depression', 'Depression'])
disp.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title("Confusion Matrix: Neural Network (MLP)")
axes[0].grid(False)

# 3. Vẽ ROC Curve
fpr, tpr, _ = roc_curve(y_val, val_probs_nn)
axes[1].plot(fpr, tpr, color='#2ca02c', lw=2, label=f'MLP Neural Network (AUC = {val_auc_nn:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].set_title('ROC Curve - Validation Set (20%)')
axes[1].legend(loc='lower right', frameon=True, shadow=True)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mlp_diagnostic_analysis.png", dpi=300)
plt.show()

# Lưu dự đoán Validation để tái sử dụng sau này (nếu cần)
np.save(f"{OUTPUT_DIR}/nn_val_probs.npy", val_probs_nn)

## 11. Tối ưu hóa ngưỡng phân loại (Threshold Optimization)

Mặc định, các mô hình phân loại thường sử dụng ngưỡng **0.5** để quyết định một mẫu thuộc lớp `0` hay `1`. Tuy nhiên, đối với dữ liệu mất cân bằng (Imbalanced Data) và các bài toán y tế, ngưỡng 0.5 thường không mang lại kết quả tốt nhất cho **F1-Score**.

Chúng ta sẽ thực hiện "vét cạn" (Grid Search) các ngưỡng từ **0.1 đến 0.9** trên tập Validation (20%) để tìm ra con số mang lại sự cân bằng hoàn hảo nhất giữa Precision và Recall, từ đó tối đa hóa điểm F1.

In [ ]:
# 1. Định nghĩa dải ngưỡng cần thử nghiệm
thresholds = np.arange(0.1, 0.91, 0.01)
f1_scores = []

# 2. Tính toán F1-Score cho từng ngưỡng trên tập Validation
for t in thresholds:
    v_preds = (val_probs_nn >= t).astype(int)
    f1_scores.append(f1_score(y_val, v_preds))

# 3. Tìm ngưỡng mang lại F1 cao nhất
opt_idx = np.argmax(f1_scores)
opt_threshold = thresholds[opt_idx]
opt_f1 = f1_scores[opt_idx]

print(f"--- KẾT QUẢ TỐI ƯU NGƯỠNG ---")
print(f"Ngưỡng tối ưu (Optimal Threshold): {opt_threshold:.2f}")
print(f"F1-Score cao nhất đạt được      : {opt_f1:.4f}")

# 4. Trực quan hóa quá trình tối ưu
plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores, color='#ff7f0e', lw=2, label='F1-Score curve')
plt.axvline(opt_threshold, color='black', linestyle='--', alpha=0.7, 
            label=f'Best Threshold: {opt_threshold:.2f}')
plt.fill_between(thresholds, f1_scores, alpha=0.1, color='#ff7f0e')
plt.title('Threshold vs F1-Score (Neural Network Optimization)', fontsize=14)
plt.xlabel('Threshold')
plt.ylabel('F1-Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12. Dự đoán tập Test & Xuất file Submission

Bước cuối cùng của Pipeline là áp dụng mô hình MLP đã được huấn luyện tốt nhất lên tập dữ liệu **Test** (`test_nn_ready.pkl`). 

**Quy trình:**
1. Tải tập dữ liệu Test đã qua tiền xử lý (cùng luồng NN).
2. Chuyển đổi sang PyTorch Tensors.
3. Dự đoán xác suất bằng mô hình `final_model`.
4. Quy đổi xác suất sang nhãn nhị phân dựa trên **Ngưỡng tối ưu ({opt_threshold:.2f})** vừa tìm được.
5. Đóng gói kết quả kèm theo `id` để tạo file nộp bài chuẩn Kaggle.

In [ ]:
# 1. Tải dữ liệu Test dành cho luồng NN
X_test_nn = pd.read_pickle(f"{INPUT_DIR}/test_nn_ready.pkl")

# Loại bỏ cột KFOLD nếu lỡ có trong tập test (để đồng nhất input_dim)
if 'KFOLD' in X_test_nn.columns:
    X_test_nn = X_test_nn.drop(columns=['KFOLD'])

# 2. Chuyển sang Tensor và đưa lên thiết bị tính toán
X_test_tensor = torch.tensor(X_test_nn.values, dtype=torch.float32).to(device)

# 3. Dự đoán (Inference mode)
final_model.eval()
with torch.no_grad():
    test_logits = final_model(X_test_tensor)
    # Áp dụng Sigmoid để lấy xác suất
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()

# 4. Áp dụng ngưỡng tối ưu để ra nhãn Depression
test_preds = (test_probs >= opt_threshold).astype(int)

# 5. Tạo file Submission
# Lưu ý: 'id' được lấy theo đúng số thứ tự dòng của tập test
submission_nn = pd.DataFrame({
    'id': range(len(test_preds)), # Đảm bảo khớp với cột id của tập test Kaggle
    'Depression': test_preds
})

# Lưu file CSV
file_name = f"{OUTPUT_DIR}/submission_neural_network.csv"
submission_nn.to_csv(file_name, index=False)

print(f"KẾT THÚC PIPELINE NEURAL NETWORK.")
print(f"Đã xuất file nộp bài: {file_name}")
display(submission_nn.head())